In [1]:
# Pandas: tabular data, alignment, and time-aware operations
import pandas as pd
import numpy as np

print("pandas:", pd.__version__)

Basics

In [2]:
import pandas as pd

# DataFrame oluşturma
df = pd.DataFrame({
    "name": ["Ali", "Ayşe", "Mehmet"],
    "age": [25, 30, 22],
    "salary": [10000, 15000, 8000]
})

# temel bakış
df.head()
df.info()
df.describe()

# kolon seçme
df["age"]

# filtreleme
df[df["age"] > 25]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   name    3 non-null      object
 1   age     3 non-null      int64 
 2   salary  3 non-null      int64 
dtypes: int64(2), object(1)
memory usage: 200.0+ bytes


,name,age,salary
1,Ayşe,30,15000


In [3]:
# Örnek satış verisi: bölge ve ürün bazında tutarlar
sales = pd.DataFrame(
    {
        "bolge": ["A", "A", "B", "B", "A"],
        "urun": ["x", "y", "x", "y", "x"],
        "tutar": [100, 150, 200, 50, 120],
    }
)

# aggregate: gruplar halinde özet istatistikler
g = sales.groupby("bolge")
print("Bölge bazında toplam tutar:\n", g["tutar"].sum())
print("\nBirden fazla agg:\n", g["tutar"].agg(["sum", "mean", "count"]))

# transform: her satırı **aynı boyutta** bırakır (ör. gruba göre normalize)
sales["tutar_grup_ort"] = sales.groupby("bolge")["tutar"].transform("mean")
sales["tutar_zscore"] = sales.groupby("bolge")["tutar"].transform(
    lambda s: (s - s.mean()) / s.std(ddof=0)
)
print("\ntransform örneği:\n", sales)

Bölge bazında toplam tutar:
 bolge
A    370
B    250
Name: tutar, dtype: int64

Birden fazla agg:
        sum        mean  count
bolge                        
A      370  123.333333      3
B      250  125.000000      2

transform örneği:
   bolge urun  tutar  tutar_grup_ort  tutar_zscore
0     A    x    100      123.333333     -1.135550
1     A    y    150      123.333333      1.297771
2     B    x    200      125.000000      1.000000
3     B    y     50      125.000000     -1.000000
4     A    x    120      123.333333     -0.162221


merge

In [8]:

customers = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Ali", "Ayşe", "Mehmet"]
})

orders = pd.DataFrame({
    "id": [1, 1, 2, 4],
    "product": ["Laptop", "Mouse", "Keyboard", "Monitor"],
    "price": [20000, 500, 1500, 7000]
})


In [9]:
#inner join
inner = pd.merge(customers, orders, on="id")
print("inner:\n", inner)

inner:
    id  name   product  price
0   1   Ali    Laptop  20000
1   1   Ali     Mouse    500
2   2  Ayşe  Keyboard   1500


In [10]:
#left join
left = pd.merge(customers, orders, on="id", how="left")
print("left:\n", left)

left:
    id    name   product    price
0   1     Ali    Laptop  20000.0
1   1     Ali     Mouse    500.0
2   2    Ayşe  Keyboard   1500.0
3   3  Mehmet       NaN      NaN


In [11]:
#right join
right = pd.merge(customers, orders, on="id", how="right")
print("right:\n", right)


right:
    id  name   product  price
0   1   Ali    Laptop  20000
1   1   Ali     Mouse    500
2   2  Ayşe  Keyboard   1500
3   4   NaN   Monitor   7000


In [12]:
#outer join
outer = pd.merge(customers, orders, on="id", how="outer")
print("outer:\n", outer)


outer:
    id    name   product    price
0   1     Ali    Laptop  20000.0
1   1     Ali     Mouse    500.0
2   2    Ayşe  Keyboard   1500.0
3   3  Mehmet       NaN      NaN
4   4     NaN   Monitor   7000.0


PIVOT TABLE

In [13]:
df = pd.DataFrame({
    "bolge": ["A", "A", "B", "B"],
    "urun": ["X", "Y", "X", "Y"],
    "satis": [100, 200, 300, 400]
})

In [16]:
df.pivot_table(
    values="satis",
    index="bolge",
    columns="urun",
    aggfunc=["mean", "max"]
)

mean         max     
urun       X      Y    X    Y
bolge                        
A      100.0  200.0  100  200
B      300.0  400.0  300  400

RESAMPLE

In [21]:
dates = pd.date_range("2024-01-01", periods=10, freq="D")

df = pd.DataFrame({
    "date": dates,
    "sales": [10,20,15,30,25,40,35,50,45,60]
})
print(df)
df.set_index("date", inplace=True)
print(df.resample("M").sum())

df.resample("M").mean()
df.resample("M").max()
df.resample("M").min()


        date  sales
0 2024-01-01     10
1 2024-01-02     20
2 2024-01-03     15
3 2024-01-04     30
4 2024-01-05     25
5 2024-01-06     40
6 2024-01-07     35
7 2024-01-08     50
8 2024-01-09     45
9 2024-01-10     60
            sales
date             
2024-01-31    330


C:\Users\hp\AppData\Local\Temp\ipykernel_12756\1201453076.py:9: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  print(df.resample("M").sum())
C:\Users\hp\AppData\Local\Temp\ipykernel_12756\1201453076.py:11: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.resample("M").mean()
C:\Users\hp\AppData\Local\Temp\ipykernel_12756\1201453076.py:12: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.resample("M").max()
C:\Users\hp\AppData\Local\Temp\ipykernel_12756\1201453076.py:13: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  df.resample("M").min()


,sales
date,
2024-01-31,10
